In [1]:
# ビックリマーク(!) ではなく パーセント(%) を使います
%pip install reversi

In [4]:
import random
import copy

# --- ここからオセロのロジック ---

def draw_board(board):
    # 盤面を表示する関数
    H_LINE = '  +---+---+---+---+---+---+---+---+'
    V_LINE = '  |   |   |   |   |   |   |   |   |'

    print('    0   1   2   3   4   5   6   7')
    print(H_LINE)
    for y in range(8):
        print(V_LINE)
        print(f'{y} ', end='')
        for x in range(8):
            print(f'| {board[x][y]} ', end='')
        print('|')
        print(H_LINE)

def get_new_board():
    # 新しい盤面を作る
    board = []
    for i in range(8):
        board.append([' '] * 8)
    return board

def is_valid_move(board, tile, xstart, ystart):
    # そこに置けるかチェックするルール判定
    if board[xstart][ystart] != ' ' or not is_on_board(xstart, ystart):
        return False

    board[xstart][ystart] = tile
    if tile == 'X':
        other_tile = 'O'
    else:
        other_tile = 'X'

    tiles_to_flip = []
    # 8方向をチェック
    for xdirection, ydirection in [[0, 1], [1, 1], [1, 0], [1, -1], [0, -1], [-1, -1], [-1, 0], [-1, 1]]:
        x, y = xstart, ystart
        x += xdirection
        y += ydirection
        if is_on_board(x, y) and board[x][y] == other_tile:
            x += xdirection
            y += ydirection
            if not is_on_board(x, y):
                continue
            while board[x][y] == other_tile:
                x += xdirection
                y += ydirection
                if not is_on_board(x, y):
                    break
            if not is_on_board(x, y):
                continue
            if board[x][y] == tile:
                while True:
                    x -= xdirection
                    y -= ydirection
                    if x == xstart and y == ystart:
                        break
                    tiles_to_flip.append([x, y])

    board[xstart][ystart] = ' '
    if len(tiles_to_flip) == 0:
        return False
    return tiles_to_flip

def is_on_board(x, y):
    return 0 <= x <= 7 and 0 <= y <= 7

def get_board_with_valid_moves(board, tile):
    # 置ける場所をヒント表示するための盤面コピーを作る
    dupe_board = copy.deepcopy(board)
    for x, y in get_valid_moves(dupe_board, tile):
        dupe_board[x][y] = '.'
    return dupe_board

def get_valid_moves(board, tile):
    # 置ける場所のリストを返す
    valid_moves = []
    for x in range(8):
        for y in range(8):
            if is_valid_move(board, tile, x, y):
                valid_moves.append([x, y])
    return valid_moves

def get_score_of_board(board):
    # スコア計算
    xscore = 0
    oscore = 0
    for x in range(8):
        for y in range(8):
            if board[x][y] == 'X':
                xscore += 1
            if board[x][y] == 'O':
                oscore += 1
    return {'X': xscore, 'O': oscore}

def make_move(board, tile, xstart, ystart):
    # 石を置いてひっくり返す
    tiles_to_flip = is_valid_move(board, tile, xstart, ystart)
    if not tiles_to_flip:
        return False
    board[xstart][ystart] = tile
    for x, y in tiles_to_flip:
        board[x][y] = tile
    return True

def get_computer_move(board, computer_tile):
    # AIの思考ルーチン（少し賢い：角を取ろうとする）
    possible_moves = get_valid_moves(board, computer_tile)
    random.shuffle(possible_moves)

    # 角（コーナー）は強いので優先して取る
    for x, y in possible_moves:
        if is_on_corner(x, y):
            return [x, y]

    # 次に、相手に最も多く取られない場所を選ぶなどのロジックを入れる場所だが
    # 今回はシンプルに、一番多くひっくり返せる場所を選ぶ
    best_score = -1
    best_move = []
    for x, y in possible_moves:
        dupe_board = copy.deepcopy(board)
        make_move(dupe_board, computer_tile, x, y)
        score = get_score_of_board(dupe_board)[computer_tile]
        if score > best_score:
            best_move = [x, y]
            best_score = score

    return best_move

def is_on_corner(x, y):
    return (x == 0 and y == 0) or (x == 7 and y == 0) or (x == 0 and y == 7) or (x == 7 and y == 7)

# --- ゲーム進行 ---

print('オセロゲームを開始します！')
print('X (あなた) vs O (コンピュータ)')

main_board = get_new_board()
# 初期配置
main_board[3][3] = 'X'
main_board[3][4] = 'O'
main_board[4][3] = 'O'
main_board[4][4] = 'X'

turn = 'player' # あなたが先攻

while True:
    if turn == 'player':
        # 人間のターン
        draw_board(main_board)
        print(f"現在のスコア: {get_score_of_board(main_board)}")
        possible_moves = get_valid_moves(main_board, 'X')

        if not possible_moves:
            print("置ける場所がありません！パスします。")
            turn = 'computer'
            continue

        print(f"置ける場所の候補: {possible_moves}")
        print("どこに置きますか？ 横 縦 の順で数字を入力してください (例: 3 2)")
        print("終了するには 'q' を入力")

        move_str = input('> ')
        if move_str.lower().startswith('q'):
            break

        try:
            x_str, y_str = move_str.split()
            x = int(x_str)
            y = int(y_str)
            if [x, y] in possible_moves:
                make_move(main_board, 'X', x, y)
                turn = 'computer'
            else:
                print("そこには置けません！もう一度。")
        except:
            print("入力エラーです。 '3 2' のように数字をスペースで区切ってください。")

    else:
        # コンピュータのターン
        print("コンピュータが考え中...", end='')
        possible_moves = get_valid_moves(main_board, 'O')
        if not possible_moves:
            print("コンピュータは置ける場所がありません。パスします。")
            turn = 'player'
            continue

        x, y = get_computer_move(main_board, 'O')
        # 少し待った演出を入れるなら time.sleep(1) を使うが、今回は即実行
        make_move(main_board, 'O', x, y)
        print(f"コンピュータは ({x}, {y}) に置きました。")
        turn = 'player'

    # ゲーム終了判定
    if not get_valid_moves(main_board, 'X') and not get_valid_moves(main_board, 'O'):
        draw_board(main_board)
        scores = get_score_of_board(main_board)
        print("ゲーム終了！")
        print(f"最終スコア - あなた(X): {scores['X']}, コンピュータ(O): {scores['O']}")
        if scores['X'] > scores['O']:
            print("あなたの勝ちです！おめでとう！")
        elif scores['X'] < scores['O']:
            print("コンピュータの勝ちです。残念！")
        else:
            print("引き分けです！")
        break

オセロゲームを開始します！
X (あなた) vs O (コンピュータ)
    0   1   2   3   4   5   6   7
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
0 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
1 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
2 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
3 |   |   |   | X | O |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
4 |   |   |   | O | X |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
5 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
6 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
  |   |   |   |   |   |   |   |   |
7 |   |   |   |   |   |   |   |   |
  +---+---+---+---+---+---+---+---+
現在のスコア: {'X': 2, 'O': 2}
置ける場所